In [1]:
import numpy as np

In [2]:
import pandas as pd

In [5]:
# df = pd.read_csv("data/2019-Oct.csv")
keep_cols = ["event_time", "event_type", "product_id", "price", "user_id", "user_session"]
df = pd.read_csv("data/2019-Oct.csv", usecols=keep_cols)

In [6]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

In [7]:
df = df.drop_duplicates()

In [8]:
df = df.dropna(subset=["user_session"]) # Drop columns whose "user_session" is NaN

In [9]:
df["event_time"] = pd.to_datetime(
    df["event_time"].str[:-3].str.strip(),
    format="%Y-%m-%d %H:%M:%S"
)

In [10]:
converted_sessions = (
    df.sort_values("event_time")
    .groupby("user_session")["event_type"]
    .last()
)

In [15]:
converted_sessions = converted_sessions[converted_sessions == "purchase"].index

AttributeError: 'Index' object has no attribute 'index'

In [12]:
df_converted = df[df["user_session"].isin(converted_sessions)]

In [13]:
diff = (
    df_converted.groupby(by="user_session")["event_time"].max()
    - df_converted.groupby(by="user_session")["event_time"].min()
)

In [14]:
df_converted["time_to_conversion"] = df_converted["user_session"].map(diff)

In [16]:
df_converted.head()

,event_time,event_type,product_id,price,user_id,user_session,time_to_conversion
280,2019-10-01 00:04:02,view,5100816,29.51,514591159,0e5dfc4b-2a55-43e6-8c05-97e1f07fbb56,0 days 00:02:00
377,2019-10-01 00:06:00,view,13800054,54.42,555332717,1dea3ee2-2ded-42e8-8e7a-4e2ad6ae942f,0 days 00:01:07
379,2019-10-01 00:06:02,purchase,5100816,29.51,514591159,0e5dfc4b-2a55-43e6-8c05-97e1f07fbb56,0 days 00:02:00
442,2019-10-01 00:07:07,purchase,13800054,54.42,555332717,1dea3ee2-2ded-42e8-8e7a-4e2ad6ae942f,0 days 00:01:07
1269,2019-10-01 02:17:27,view,15100009,321.73,515519720,733826b5-64a3-49a1-9c22-38efb66f8839,0 days 00:27:33


In [23]:
dropped_sessions = (
    df.sort_values("event_time")
    .groupby("user_session")["event_type"]
    .last()
)

In [25]:
dropped_sessions = dropped_sessions[dropped_sessions != "purchase"].index

In [26]:
df_dropped = df[df["user_session"].isin(dropped_sessions)]

In [27]:
num_of_dropped_sessions = len(df_dropped.groupby(by="user_session"))

In [28]:
num_of_converted_sessions = len(df_converted.groupby(by="user_session"))

In [29]:
total_sessions = num_of_dropped_sessions + num_of_converted_sessions

In [30]:
drop_rate = num_of_dropped_sessions / total_sessions * 100

In [31]:
print(drop_rate)

96.85043552213817


In [36]:
df.head()

,event_time,event_type,product_id,price,user_id,user_session
0,2019-10-01 00:00:00,view,44600062,35.79,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
1,2019-10-01 00:00:00,view,3900821,33.20,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc
2,2019-10-01 00:00:01,view,17200506,543.10,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8
3,2019-10-01 00:00:01,view,1307067,251.74,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713
4,2019-10-01 00:00:04,view,1004237,1081.98,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d


In [40]:
purchased_products = (
    df.sort_values("event_time")
    .groupby(["user_id", "product_id"])["event_type"]
    .last()
)

In [41]:
purchased_products = purchased_products[purchased_products == "purchase"].index

In [42]:
df_purchased = df[
    df.set_index(["user_id", "product_id"]).index.isin(purchased_products)
]

In [43]:
df_purchased.head()

,event_time,event_type,product_id,price,user_id,user_session
23,2019-10-01 00:00:23,view,1004739,197.55,519530528,9882d21f-2c5f-496b-90d4-a1503edb6562
152,2019-10-01 00:02:08,view,1002528,643.23,551377651,3c80f0d6-e9ec-4181-8c5c-837a30be2d68
309,2019-10-01 00:04:37,view,1004209,92.64,520775875,a653ed61-904a-4804-bb87-f8e87443350a
322,2019-10-01 00:05:01,view,1005115,975.57,529755884,0b828fb6-99bd-4d26-beb3-3021f5d6102c
365,2019-10-01 00:05:48,view,1005130,1619.09,544647372,596ff182-8e6b-4b27-bb55-1dc66cfb6bb6


In [48]:
df_purchased.loc[df['product_id'] == 1004739].loc[df['user_id'] == 519530528]

,event_time,event_type,product_id,price,user_id,user_session
23,2019-10-01 00:00:23,view,1004739,197.55,519530528,9882d21f-2c5f-496b-90d4-a1503edb6562
502572,2019-10-01 10:25:04,view,1004739,197.43,519530528,eccf2498-9c1d-49a6-b577-9cf5092621cb
709646,2019-10-01 13:38:11,view,1004739,197.43,519530528,ce156cc8-00c9-4ed3-8d7c-383279dbefc1
711493,2019-10-01 13:39:51,view,1004739,197.43,519530528,ce156cc8-00c9-4ed3-8d7c-383279dbefc1
1280595,2019-10-02 02:14:58,view,1004739,197.30,519530528,bf025e8f-ab2d-4a04-a6c8-92663493ca56
1280844,2019-10-02 02:15:28,view,1004739,197.30,519530528,bf025e8f-ab2d-4a04-a6c8-92663493ca56
1499251,2019-10-02 06:28:12,view,1004739,197.17,519530528,fe016254-1f6f-4985-a2b7-1ea2412a5c43
1739457,2019-10-02 10:03:35,view,1004739,197.17,519530528,9e68a8a8-4ab9-4363-a8b4-bedc97655641
1739703,2019-10-02 10:03:49,view,1004739,197.12,519530528,9e68a8a8-4ab9-4363-a8b4-bedc97655641
1740079,2019-10-02 10:04:09,view,1004739,197.12,519530528,9e68a8a8-4ab9-4363-a8b4-bedc97655641


In [49]:
df_purchased["time_to_conversion"] = (
    df_purchased["event_time"] 
    - df_purchased.groupby(["user_id", "product_id"])["event_time"].transform("min")
)

In [51]:
df_purchased["time_to_conversion"] = df_purchased["time_to_conversion"].dt.total_seconds()

In [56]:
df_purchased.groupby(["product_id"])["time_to_conversion"].mean()

product_id
1000978         39.000000
1001588        616.000000
1002042         62.833333
1002062     112942.352941
1002098     135627.444444
                ...      
58300005       166.500000
58300012       163.666667
59000004    145996.571429
60500001        96.333333
60500002       188.086957
Name: time_to_conversion, Length: 25457, dtype: float64

In [ ]:
s = df_purchased.groupby(["product_id"])["time_to_conversion"].mean().describe()

count    2.545700e+04
mean     7.913406e+04
std      1.758969e+05
min      0.000000e+00
25%      1.080000e+02
50%      1.633500e+03
75%      7.801352e+04
max      2.163577e+06
Name: time_to_conversion, dtype: float64

In [61]:
df_purchased.groupby(['product_id']).size()

product_id
1000978      2
1001588     12
1002042      6
1002062     34
1002098      9
            ..
58300005     2
58300012     6
59000004    14
60500001     3
60500002    23
Length: 25457, dtype: int64

In [63]:
avg_conversion_time = df_purchased.groupby(["product_id"])["time_to_conversion"].mean()
total_purchases = df_purchased.groupby(["product_id"]).size()

In [67]:
df_metrics = pd.concat(
    [avg_conversion_time, total_purchases], axis=1
).reset_index()
df_metrics.columns = ["product_id", "avg_conversion_time", "total_purchases"]

In [68]:
df_metrics.head()

,product_id,avg_conversion_time,total_purchases
0,1000978,39.000000,2
1,1001588,616.000000,12
2,1002042,62.833333,6
3,1002062,112942.352941,34
4,1002098,135627.444444,9


In [70]:
df_metrics["product_id"].unique()

array([ 1000978,  1001588,  1002042, ..., 59000004, 60500001, 60500002],
      shape=(25457,))

In [71]:
len(df_metrics["product_id"])

25457